## Calculating the Units for Multi-family Homes across California

This notebook utilizes parcel data, Zillow data, and building data to calculate missing multi-family unit data. Residential parcels are subset to calculate building volume and create a regression that gives best approximations of multi-family home data. Single family home volume is additionally calculated. This unit data is utilized in calculating hosting capacity to assess where limited distributed energy resources exist across California.



In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

In [2]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

#### Investigate parcel data

The analysis relies on parcels to calculate which buildings are assigned to which units. Some buildings intersect with more than one parcel which is kept in both calculations would duplicate the volume of thse homes and cause erroneous unit calculations. Hence a unique parcel number is required to ensure that the calculations accurate.

In [3]:
# load parcels 
parcels = gpd.read_parquet("/../../capstone/electrigrid/data/parcels/parcels_final.parquet").to_crs(epsg=4326)

In [4]:
# investigate duplicated parcel numbers
parcels[parcels['PARNO'].duplicated()]


,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry
2280,482-80-1-6,Alameda,None,None,None,172.340340,1242.904591,"MULTIPOLYGON Z (((-122.09155 37.59110 0.00000,..."
2413,941-2792-176,Alameda,None,None,None,1988.452430,6015.218730,"MULTIPOLYGON Z (((-121.91067 37.72566 0.00000,..."
2543,941-2792-176,Alameda,None,None,None,59.652862,92.678335,"MULTIPOLYGON Z (((-121.91158 37.72673 0.00000,..."
2711,946-4569-207-1,Alameda,None,None,None,75.019840,266.873353,"MULTIPOLYGON Z (((-121.84896 37.66600 0.00000,..."
5879,941-2792-176,Alameda,None,None,None,597.983026,1798.667011,"MULTIPOLYGON Z (((-121.91074 37.72763 0.00000,..."
...,...,...,...,...,...,...,...,...
11532333,common,Ventura,None,None,None,NaN,NaN,"POLYGON Z ((-119.06553 34.37366 0.00000, -119...."
11532334,common,Ventura,None,None,None,NaN,NaN,"POLYGON Z ((-119.05863 34.37424 0.00000, -119...."
11532335,common,Ventura,None,None,None,NaN,NaN,"POLYGON Z ((-118.83605 34.17332 0.00000, -118...."
11532336,common,Ventura,None,None,None,NaN,NaN,"POLYGON Z ((-118.83327 34.14923 0.00000, -118...."


The parcel number should have been unique to each parcel. As shown above there are 525,130 parcel numbers that are not unique. Instead we'll assume that each row is a different parcel and assign a unique id from the index. 

In [4]:
# make an id column in the parcel dataframe to use as the unique id
parcels['parcel_ID'] = parcels.index

In [6]:
parcels.head()

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,parcel_ID
0,48-6298-3-2,Alameda,None,None,None,103.683187,606.327284,"MULTIPOLYGON Z (((-122.12384 37.75571 0.00000,...",0
1,48-6313-23,Alameda,None,None,None,148.045063,1083.135315,"MULTIPOLYGON Z (((-122.12504 37.75429 0.00000,...",1
2,48-6313-87,Alameda,None,None,None,97.520632,557.505860,"MULTIPOLYGON Z (((-122.12635 37.75366 0.00000,...",2
3,48-6330-8-2,Alameda,None,None,None,171.271160,1602.682949,"MULTIPOLYGON Z (((-122.12243 37.75165 0.00000,...",3
4,48-6432-32,Alameda,None,None,None,140.208009,1078.107621,"MULTIPOLYGON Z (((-122.12829 37.76110 0.00000,...",4


In [5]:
# import Zillow data 
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

In [6]:
zillow.shape

(10012620, 16)

We expect to get the same number of homes at the end of unit regression calculation.

Building Footprint

Access: https://sat-io.earthengine.app/view/gba

In [7]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

# Unit-regression for multi-family homes and volume calculations for all homes

Start by selecting all of the multi-family homes. 


In [49]:
# the analysis only requires the parcel_id and geometry  
parcels = parcels[['parcel_ID', 'geometry']]

## SUBSET ALL THE DATA TYPES
## select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

# select the single family home data 
single_zillow = zillow[zillow['type'] == "Single"]

# select the condo data 
zillow_condos = zillow[(zillow['code'] == "RR106") & (zillow['code'] != 'Single' )]

In [ ]:
# check to ensure all of the subsets are accurate, these all add up to the total of the zillow 
zillow_multi.shape[0] + single_zillow.shape[0] + zillow_condos.shape[0]

10012620

In [50]:

## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi, predicate="contains").index.unique()

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects").index.unique()
buildings_res = building.loc[valid_buildings]

## join parcels to buildings (keeping observations as parcels, but with building attributes)
# sum number of units per parcel
units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
parcels_res = parcels_res.join(units_sum)

# confirm that joining with Zillow decreased the number of buildings
assert len(buildings_res) < len(building)

# keep all residential buildings, and add zillow points only where they match up
building_zillow = gpd.sjoin(
    buildings_res,
    zillow_multi,
    predicate = "intersects")

# drop the unnecessary index
building_zillow = building_zillow.drop(columns = "index_right")

# add in the parcel number to add all of the building volumes for the regression
building_zillow_parcels = building_zillow.sjoin(
                                parcels_res,
                                how = "left",
                                predicate = "intersects")

## FOR BUILDINGS INTERSECTING MORE THAN ONE PARCEL CALCULATE WHICH PARCEL IT INTERSECTS MORE
# change the crs to a projected crs
building_zillow_parcels = building_zillow_parcels.to_crs("EPSG:6933")
parcels_res = parcels_res.to_crs("EPSG:6933")

In [ ]:
print(f"There are ",building_zillow_parcels['parcel_ID'].isna().sum(), "buildings not associated with a parcel.")

# check the number of zillow points that don't have parcels associated with them but have units
print(f"Of all of the homes that are not within parcels", building_zillow_parcels[building_zillow_parcels['parcel_ID'].isna()]['unit_left'].isna().sum(), " also do not have units.")

There are  438 buildings not associated with a parcel.
Of all of the homes that are not within parcels 426 are also do not have units.


These need to be assigned to the nearest building. The volume for each of these buildings can be utilized to calculate volume. The volume can then be utilized to 

In [ ]:
# view the buildings without parcels 
building_zillow_parcels[building_zillow_parcels['parcel_ID'].isna()]

,source,id,height,var,region,bbox,geometry,type,year,room,heat,cool,own,unit_left,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,index_right,parcel_ID,unit_right
26253,osm,45594310,5.411443,0.821852,USA,"{'xmax': -120.6776363, 'xmin': -120.6778611999...","POLYGON ((-11643755.041 4229423.394, -11643746...",Multi,NaN,3.0,None,None,None,NaN,815421.0,living,1313.0,8935955,06079011202,219,PGE/SCE,RI110,NaN,NaN,NaN
27745,osm,219426397,1.806054,0.751210,USA,"{'xmax': -120.65774779999998, 'xmin': -120.657...","POLYGON ((-11641822.614 4228764.404, -11641817...",Multi,NaN,2.0,None,None,None,NaN,195312.0,living,994.0,8878855,06079011101,211,PGE/SCE,RI110,NaN,NaN,NaN
27747,osm,219426395,3.997953,0.871557,USA,"{'xmax': -120.65796530000001, 'xmin': -120.658...","POLYGON ((-11641847.208 4228753.473, -11641838...",Multi,NaN,3.0,None,None,O,NaN,636902.0,living,1395.0,8878853,06079011101,211,PGE/SCE,RI110,NaN,NaN,NaN
27753,osm,219426365,4.353393,0.295156,USA,"{'xmax': -120.65719900000002, 'xmin': -120.657...","POLYGON ((-11641784.328 4228692.909, -11641781...",Multi,NaN,4.0,None,None,O,NaN,564752.0,living,2807.0,8878916,06079011101,211,PGE/SCE,RI110,NaN,NaN,NaN
27762,osm,219426423,4.539310,1.343797,USA,"{'xmax': -120.6586921, 'xmin': -120.6588903999...","POLYGON ((-11641916.630 4228720.681, -11641911...",Multi,NaN,6.0,None,None,None,NaN,586555.0,living,1452.0,8878858,06079011101,211,PGE/SCE,RI102,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170461,osm,170046583,3.389146,0.528725,USA,"{'xmax': -120.6951085, 'xmin': -120.6953514999...","POLYGON ((-11645444.603 4276932.745, -11645422...",Multi,NaN,NaN,None,None,None,NaN,938173.0,None,NaN,8909511,06079010016,334,PGE/SCE,RI104,NaN,NaN,NaN
170681,osm,170047773,3.772871,0.264081,USA,"{'xmax': -120.68197969999997, 'xmin': -120.682...","POLYGON ((-11644189.654 4276516.503, -11644155...",Multi,NaN,NaN,None,None,O,NaN,229647.0,None,NaN,8916586,06079010016,334,PGE/SCE,RI110,NaN,NaN,NaN
172195,osm,196918869,2.470065,1.030752,USA,"{'xmax': -120.35142850000001, 'xmin': -120.351...","POLYGON ((-11612266.831 4268484.382, -11612286...",Multi,NaN,3.0,None,None,None,NaN,381523.0,living,1728.0,8995095,06079010301,19,PGE/SCE,RI109,NaN,NaN,NaN
173756,osm,204562919,0.581746,0.080596,USA,"{'xmax': -120.45206599999999, 'xmin': -120.452...","POLYGON ((-11621989.097 4252625.326, -11621979...",Multi,NaN,3.0,others,None,None,NaN,320163.0,living,1785.0,8929187,06079010301,633,PGE/SCE,RI109,NaN,NaN,NaN


In [51]:
# drop rows with no parcel match before computing intersection
building_zillow_parcels = building_zillow_parcels.dropna(subset=['index_right'])
building_zillow_parcels['index_right'] = building_zillow_parcels['index_right'].astype(int)

# calculate intersection area for each building-parcel pair
building_zillow_parcels["intersection_area"] = (
    building_zillow_parcels.geometry.values
    .intersection(parcels_res.loc[building_zillow_parcels["index_right"]].geometry.values).area)

# keep only the parcel with the largest overlap per building
building_zillow_parcels = (
    building_zillow_parcels
    .sort_values("intersection_area", ascending=False)
    .loc[~building_zillow_parcels.index.duplicated(keep="first")]
    .drop(columns="intersection_area"))

# drop the unnecessary columns
building_zillow_parcels = building_zillow_parcels.drop(columns = ['index_right', 'unit_left'])
building_zillow_parcels = building_zillow_parcels.rename(columns = {'unit_right': 'total_unit'})

In [52]:


# reproject data frame to crs with meters as units
building_m = building_zillow_parcels.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

# keep only observations with unit data
building_w_units = building_m[~building_m['total_unit'].isna()]

assert building_w_units['total_unit'].isna().sum() == 0

# drop multi-family homes where the total_unit is equal to zero
building_w_units = building_w_units.drop(building_w_units[building_w_units['total_unit'] == 0].index)

# aggregate the volumes by parcel
total_volume_m3 = building_w_units.groupby('parcel_ID')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
total_volume_m3 = pd.DataFrame(total_volume_m3)

# rename the column to total_volume_m3
total_volume_m3 = total_volume_m3.rename(columns = {'volume_m3' : 'total_volume_m3'})

# add the total volume to the buildings dataframe
building_w_units = building_w_units.join(total_volume_m3, on = 'parcel_ID')

# some of the homes don't have a parcel number (so replace the total volume with volume if its na
building_w_units['total_volume_m3'] = building_w_units['total_volume_m3'].fillna(building_w_units['volume_m3'])

# remove duplicates from the parcel number so that it doesn't skew the linear regression
building_w_units = building_w_units.drop_duplicates(subset = 'parcel_ID', keep = 'first')

# run model
results = smf.ols('total_unit ~ total_volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('total_unit ~ total_volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

# extract just the multi-family homes where unit info is missing
missing_units = building_m[(building_m['total_unit'].isna()) | (building_m['total_unit'] == 0)]


# aggregate the volumes for the unit regression by parcel
missing_total_volume_m3 = missing_units.groupby('ID')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
missing_total_volume_m3 = pd.DataFrame(missing_total_volume_m3)

# rename the column to missing_total_volume_m3
missing_total_volume_m3 = missing_total_volume_m3.rename(columns = {'volume_m3' : 'missing_total_volume_m3'})

# add the missing total volume to the buildings dataframe
missing_units = missing_units.join(missing_total_volume_m3, on = 'parcel_ID')

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace anywhere the missing_total_volume is missing (fill in for the outliers since total_volume was computed before)
missing_outlier_units['missing_total_volume_m3'] = missing_outlier_units['missing_total_volume_m3'].fillna(missing_outlier_units['total_volume_m3'])

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('total_unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['total_unit'] = round(intercept + missing_outlier_units_pred['missing_total_volume_m3'] * slope)

## multiple buildings in one parcel have duplicated units since they were computed on total volume
#  only keep the first observation per parcel
missing_outlier_units_pred = missing_outlier_units_pred.drop_duplicates(subset = 'parcel_ID', keep = 'first')

# combine multi-family homes data frames
multi_complete = pd.concat([building_units_clean, missing_outlier_units_pred]).to_crs(zillow.crs)

# fill the total_volume for those with the missing_total_volume_m3 and rename to total_volume_m3
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# fill the total_volume for those with the missing_total_volume_m3 
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# drop unnecessary columns 
multi_complete = multi_complete.drop(['residual', 'geometry', 'total_volume_m3'], axis = 1)

# rename to total_volume_m3
multi_complete = multi_complete.rename(columns = {'missing_total_volume_m3' : 'total_volume_m3'})

# drop the unit data from parcel res as the new unit data was computed
parcels_res = parcels_res.drop(columns = ['unit'])


/tmp/ipykernel_3126263/306878399.py:55: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept = results_clean.params[0]
/tmp/ipykernel_3126263/306878399.py:56: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = results_clean.params[1]


In [55]:
parcels_res.head()


,parcel_ID,geometry
92,92,"MULTIPOLYGON Z (((-122.11883 37.75180 0.00000,..."
212,212,"MULTIPOLYGON Z (((-122.17728 37.72609 0.00000,..."
231,231,"MULTIPOLYGON Z (((-122.12955 37.75105 0.00000,..."
300,300,"MULTIPOLYGON Z (((-122.25473 37.81432 0.00000,..."
318,318,"MULTIPOLYGON Z (((-122.28456 37.81261 0.00000,..."


In [56]:

# join the parcel data on ID to get the parcel geometry
multi_by_parcel = parcels_res.merge(multi_complete, on = 'parcel_ID')

## IN ORDER TO CONCAT THE MULTI-FAMILY HOMES WITH SINGLE FAMILY HOMES THEY MUST ALL BE POINTS
# set the geometry to the parcel geometry
multi_by_parcel = multi_by_parcel.set_geometry('geometry')

# change the crs 
multi_by_parcel = multi_by_parcel.to_crs("EPSG:6933")

# create centroids for each multi residential parcel 
multi_by_parcel['centroids'] = multi_by_parcel.geometry.centroid

# drop the geometry of multi_by_parcel so the centroid becomes the geometry
multi_by_parcel = multi_by_parcel.drop(columns = 'geometry')

# rename the centroid to geometry
multi_by_parcel = multi_by_parcel.rename(columns = {'centroids' : 'geometry'})

# set the centroid to the geometry
multi_by_parcel_processed = multi_by_parcel.set_geometry('geometry')

# change the CRS back to the Zillow CRS 
multi_by_parcel_processed = multi_by_parcel_processed.to_crs(zillow.crs)


In [58]:
multi_by_parcel_processed.head()

,parcel_ID,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,geometry
0,92,osm,470801986,7.360551,3.760925,USA,"{'xmax': -122.11905469999998, 'xmin': -122.119...",Multi,1970.0,3.0,None,None,O,1447455.0,living,2621.0,135866,06001409900,262,PGE/SCE,RI101,2.0,344.514287,2535.814937,2535.814937,POINT (-122.11915 37.75188)
1,212,osm,467433407,3.687969,0.562078,USA,"{'xmax': -122.1772724, 'xmin': -122.1774869999...",Multi,1965.0,6.0,None,None,I,518707.0,living,3038.0,113661,06001409200,568,PGE/SCE,RI103,4.0,180.790440,666.749620,666.749620,POINT (-122.17738 37.72601)
2,231,osm,471260067,12.743473,1.726028,USA,"{'xmax': -122.12957120000002, 'xmin': -122.129...",Multi,1955.0,7.0,None,None,O,1335766.0,living,4473.0,135372,06001410000,262,PGE/SCE,RI101,2.0,452.441520,5765.676322,5765.676322,POINT (-122.12977 37.75103)
3,300,osm,308164383,7.269603,2.313044,USA,"{'xmax': -122.25476140000002, 'xmin': -122.255...",Multi,1966.0,36.0,None,None,None,8022880.0,living,24529.0,3702,06001403600,574,PGE/SCE,RI104,26.0,894.878022,6505.407751,6505.407751,POINT (-122.25494 37.81419)
4,318,osm,473407799,5.020492,0.822397,USA,"{'xmax': -122.2846471, 'xmin': -122.2848646, '...",Multi,1905.0,4.0,None,None,None,270454.0,living,2625.0,162813,06001402400,468,PGE/SCE,RI101,2.0,165.396340,830.370936,830.370936,POINT (-122.28473 37.81260)


In [59]:
# takes a really long time 
multi_by_parcel_processed.to_parquet("multi_by_parcel_processed.parquet")

In [ ]:


## SINGLE FAMILY UNITS 

# # duplicate the geometry 
# single_zillow['zillow_geometry'] = single_zillow['geometry']

# ## CALCULATE AREA FOR SINGLE FAMILY HOME
# # select parcels where single family homes exist 
# single_parcels = parcels.sjoin(single_zillow, predicate="contains").index.unique()
# single_parcels = parcels.loc[single_parcels]

# # crop to residential buildings (by keeping only those within single residential parcels)
# single_buildings = building.sjoin(single_parcels, predicate="intersects").index.unique()
# buildings_res = building.loc[single_buildings]

# # keep all single-family buildings, and add zillow points only where they match up
# single_building_zillow = gpd.sjoin(
#     buildings_res,
#     single_zillow,
#     predicate = "intersects")

# # drop unnecessary columns
# single_building_zillow = single_building_zillow.drop(columns = "index_right")

# # reproject data frame to crs with meters as units
# single_building_zillow = single_building_zillow.to_crs("EPSG:6933")

# # create column from polygon area
# single_building_zillow['area_m2'] = single_building_zillow.area

# # rename height column to be clear about units
# single_building_zillow.rename(columns={"height":"height_m"}, inplace = True)

# # create volume column
# single_building_zillow['volume_m3'] = single_building_zillow['area_m2'] * single_building_zillow['height_m']

# # change the crs back to projected
# single_building_zillow = single_building_zillow.to_crs(epsg=4326)

# # set geometry to the zillow points 
# single_building_zillow = single_building_zillow.set_geometry('zillow_geometry')


/Users/sofiarodas/.conda/envs/electrigrid-env/lib/python3.11/site-packages/geopandas/geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [ ]:

# assert zillow_condos.crs == complete_zillow.crs 

# # add the condo data to the complete zillow data
# complete_zillow = pd.concat([complete_zillow, zillow_condos], axis = 0)

# Final results 

## Renaming and saving